# Distance euclidienne et normalisation — Exemple k-NN à 2 descripteurs

Ce notebook illustre, pas à pas, pourquoi la **normalisation est indispensable** avant
le calcul de la distance euclidienne dans un classifieur k-NN.

**Contexte :** classification du mode de transport (Arrêt / Camion / Train / Avion)
à partir de l'accélération ICM-20948.  
**Exemple simplifié** : 2 descripteurs seulement — $g_\text{rms}$ (g) et centroïde $f_c$ (Hz) —
pour que le résultat soit visualisable en 2D.

**Plan :**
1. Données d'entraînement (3 classes)
2. Distance euclidienne **sans** normalisation → résultat incorrect
3. Normalisation centrage-réduction
4. Distance euclidienne **avec** normalisation → résultat correct
5. Visualisation côte à côte
6. Extension à 6 descripteurs (cas réel du dossier)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'

## 1. Données d'entraînement

Trois fenêtres de 10 s représentent les trois classes. Chacune est décrite par :
- $g_\text{rms}$ : valeur efficace de l'accélération (g)
- $f_c$ : centroïde spectral (Hz) = $\sum_f f \cdot S(f) / \sum_f S(f)$

Un quatrième vecteur **inconnu** doit être classé.

In [ ]:
# Descripteurs : [g_rms (g), centroide f_c (Hz)]
classes  = ['Arret', 'Camion', 'Train']
colors   = ['#7f8c8d', '#c0392b', '#2471a3']

X_train = np.array([
    [0.004,  3.0],   # Arret  : tres faible niveau, basse frequence
    [0.400, 12.0],   # Camion : niveau eleve, profil ASTM 4-50 Hz
    [0.100,  9.0],   # Train  : niveau moyen, pic ferroviaire ~8-10 Hz
])
y_train = np.array([0, 1, 2])   # labels

x_inconnu = np.array([0.38, 8.5])   # vecteur a classer

print('Jeu d entraînement :')
print(f'  {"Classe":<8}  {"g_rms (g)":>10}  {"f_c (Hz)":>10}')
for i, (row, cls) in enumerate(zip(X_train, classes)):
    print(f'  {cls:<8}  {row[0]:>10.3f}  {row[1]:>10.1f}')
print(f'\nVecteur inconnu x = (g_rms={x_inconnu[0]} g, f_c={x_inconnu[1]} Hz)')

## 2. Distance euclidienne sans normalisation

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{(a_1 - b_1)^2 + (a_2 - b_2)^2}$$

On calcule la distance entre le vecteur inconnu et chacune des 3 classes.

In [ ]:
def distance_euclidienne(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

print('Distances euclidiennes SANS normalisation :')
print(f'  {"Classe":<8}  {"Calcul detaille":<45}  {"Distance":>8}')

dist_brut = []
for row, cls in zip(X_train, classes):
    d_grms = (x_inconnu[0] - row[0]) ** 2
    d_fc   = (x_inconnu[1] - row[1]) ** 2
    d      = np.sqrt(d_grms + d_fc)
    dist_brut.append(d)
    detail = f'sqrt({d_grms:.4f} + {d_fc:.4f})'
    print(f'  {cls:<8}  {detail:<45}  {d:>8.3f}')

pred_brut = classes[np.argmin(dist_brut)]
print(f'\n  --> Prediction SANS normalisation : {pred_brut}  (distance min = {min(dist_brut):.3f})')
print(f'  --> Attendu : Camion (g_rms = {x_inconnu[0]} g est typique du camion)')
print()
print('  REMARQUE : le terme (8.5 - 9)^2 = 0.25 en Hz^2 et le terme (0.38 - 0.10)^2 = 0.078 en g^2')
print('  sont dans des unités différentes — leur somme n\'a pas de sens physique.')
print('  L\'axe f_c (plage ~10 Hz) domine l\'axe g_rms (plage ~0.4 g) et fausse le résultat.')

## 3. Normalisation centrage-réduction

Pour chaque descripteur $j$ :

$$\hat{x}_j = \frac{x_j - \mu_j}{\sigma_j}$$

où $\mu_j$ et $\sigma_j$ sont calculés **sur le jeu d'entraînement uniquement**,
puis appliqués au vecteur inconnu avec les **mêmes** paramètres.

In [ ]:
mu    = X_train.mean(axis=0)
sigma = X_train.std(axis=0)    # ecart-type population (ddof=0)

print('Parametres de normalisation (calcules sur le jeu d entraînement) :')
print(f'  mu    = (g_rms={mu[0]:.3f} g, f_c={mu[1]:.2f} Hz)')
print(f'  sigma = (g_rms={sigma[0]:.3f} g, f_c={sigma[1]:.2f} Hz)')
print()

X_norm      = (X_train - mu) / sigma
x_inc_norm  = (x_inconnu - mu) / sigma

print('Vecteurs normalises :')
print(f'  {"Classe":<8}  {"g_rms_norm":>12}  {"fc_norm":>10}')
for row_n, cls in zip(X_norm, classes):
    print(f'  {cls:<8}  {row_n[0]:>12.3f}  {row_n[1]:>10.3f}')
print(f'  {"x inconnu":<8}  {x_inc_norm[0]:>12.3f}  {x_inc_norm[1]:>10.3f}')

## 4. Distance euclidienne avec normalisation

In [ ]:
print('Distances euclidiennes AVEC normalisation :')
print(f'  {"Classe":<8}  {"Calcul detaille":<55}  {"Distance":>8}')

dist_norm = []
for row_n, cls in zip(X_norm, classes):
    d_grms = (x_inc_norm[0] - row_n[0]) ** 2
    d_fc   = (x_inc_norm[1] - row_n[1]) ** 2
    d      = np.sqrt(d_grms + d_fc)
    dist_norm.append(d)
    detail = f'sqrt({d_grms:.3f} + {d_fc:.3f})'
    print(f'  {cls:<8}  {detail:<55}  {d:>8.3f}')

pred_norm = classes[np.argmin(dist_norm)]
print(f'\n  --> Prediction AVEC normalisation : {pred_norm}  (distance min = {min(dist_norm):.3f})')
print()

print('Comparaison :')
print(f'  {"Classe":<8}  {"Sans norm":>10}  {"Avec norm":>10}')
for cls, d_b, d_n in zip(classes, dist_brut, dist_norm):
    print(f'  {cls:<8}  {d_b:>10.3f}  {d_n:>10.3f}')

## 5. Visualisation côte à côte

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('k-NN (k=1) : impact de la normalisation sur le résultat', fontsize=12)

datasets = [
    (X_train,    x_inconnu,   'Sans normalisation', 'g_rms (g)', 'Centroïde f_c (Hz)'),
    (X_norm,     x_inc_norm,  'Avec normalisation', 'g_rms (normalisé)', 'f_c (normalisée)'),
]
preds = [pred_brut, pred_norm]

for ax, (X, x_q, titre, xl, yl), pred, dist_list in zip(
        axes, datasets, preds, [dist_brut, dist_norm]):

    # Points d'entraînement
    for i, (row, cls, col) in enumerate(zip(X, classes, colors)):
        ax.scatter(row[0], row[1], s=180, color=col, zorder=5,
                   label=f'{cls}  (d={dist_list[i]:.3f})')
        ax.annotate(cls, (row[0], row[1]),
                    textcoords='offset points', xytext=(8, 6), fontsize=9)

    # Vecteur inconnu
    ax.scatter(x_q[0], x_q[1], s=220, marker='*', color='gold',
               edgecolors='k', linewidths=0.8, zorder=6, label='x inconnu')
    ax.annotate('x inconnu', (x_q[0], x_q[1]),
                textcoords='offset points', xytext=(8, -14), fontsize=9)

    # Segments vers chaque voisin
    for row, col in zip(X, colors):
        ax.plot([x_q[0], row[0]], [x_q[1], row[1]],
                color=col, lw=1.2, ls='--', alpha=0.6)

    # Mise en valeur du voisin le plus proche
    idx_min = np.argmin(dist_list)
    ax.plot([x_q[0], X[idx_min, 0]], [x_q[1], X[idx_min, 1]],
            color=colors[idx_min], lw=2.5, ls='-', alpha=0.9, zorder=4)

    ax.set_xlabel(xl, fontsize=10)
    ax.set_ylabel(yl, fontsize=10)
    ax.set_title(f'{titre}\n→ Prédiction : {pred}', fontsize=10,
                 color='green' if pred == 'Camion' else 'red')
    ax.legend(fontsize=8, loc='best')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Pourquoi l'axe f_c écrase g_rms sans normalisation

Décomposons la distance au carré en deux contributions :

In [ ]:
print('Contribution de chaque axe dans d² (sans normalisation) :')
print(f'  {"Classe":<8}  {"(Δg_rms)²":>12}  {"(Δf_c)²":>12}  {"% axe f_c":>12}')
for row, cls in zip(X_train, classes):
    dg2 = (x_inconnu[0] - row[0]) ** 2
    df2 = (x_inconnu[1] - row[1]) ** 2
    pct = 100 * df2 / (dg2 + df2) if (dg2 + df2) > 0 else 0
    print(f'  {cls:<8}  {dg2:>12.5f}  {df2:>12.4f}  {pct:>11.1f} %')

print()
print('→ L\'axe f_c (en Hz, plage ≈ 10) représente plus de 95 % de la distance au carré.')
print('  Le g_rms (en g, plage ≈ 0,4) est rendu invisible.')
print()
print('  Après normalisation, les deux axes sont réduits à σ = 1 :')
print(f'  sigma(g_rms) = {sigma[0]:.3f} g  →  écart normalisé ≈ 1 unité pour une variation typique')
print(f'  sigma(f_c)   = {sigma[1]:.2f} Hz →  écart normalisé ≈ 1 unité pour une variation typique')

## 7. Extension au cas réel — 6 descripteurs

Dans le dossier (ICM-20948, 4 modes : Arrêt / Camion / Train / Avion), chaque fenêtre
est représentée par un vecteur dans $\mathbb{R}^6$ :

| Descripteur | Définition | Bande |
|---|---|---|
| $g_\text{rms}$ | $\sqrt{\frac{1}{N}\sum x^2}$ | — |
| Crête | $\max|x| / g_\text{rms}$ | — |
| $E_\text{basse}$ | Énergie relative du périodogramme | 1–5 Hz |
| $E_\text{milieu}$ | Énergie relative du périodogramme | 5–15 Hz |
| $E_\text{haute}$ | Énergie relative du périodogramme | 15–25 Hz |
| Centroïde $f_c$ | $\sum_f f \cdot S(f) / \sum_f S(f)$ | 1–25 Hz |

Le même raisonnement s'applique en dimension 6 :

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_{j=1}^{6} \left(\hat{a}_j - \hat{b}_j\right)^2}$$

où $\hat{a}_j$ et $\hat{b}_j$ sont les composantes **normalisées**.

In [ ]:
# Demonstration rapide avec des donnees synthetiques (k=3, 4 classes)
from scipy.signal import welch
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

np.random.seed(42)
FS, T_WIN = 50.0, 10.0
N_WIN = int(T_WIN * FS)
MODES_4 = ['Arret', 'Camion', 'Train', 'Avion']

def gen_signal(mode, N):
    if mode == 0:   # Arret
        return 0.003 * np.random.randn(N)
    elif mode == 1: # Camion
        freqs = np.fft.rfftfreq(N, 1/FS)
        asd = np.zeros_like(freqs)
        m = freqs >= 4
        asd[m] = 2e-3 * (freqs[m] / 4) ** (-1.0)
        sp = np.sqrt(asd * N / (2*FS)) * np.exp(1j * 2*np.pi*np.random.rand(len(freqs)))
        x = np.fft.irfft(sp, N)
        return x * (0.40 / (x.std() + 1e-12))
    elif mode == 2: # Train
        t = np.arange(N) / FS
        fj = 7.5 + 1.5 * np.random.rand()
        x = 0.08*np.sin(2*np.pi*fj*t) + 0.04*np.sin(2*np.pi*2*fj*t)
        idx = (np.arange(int(T_WIN*fj)) * FS/fj).astype(int)
        idx = idx[idx < N]
        x[idx] += 0.3 * np.random.randn(len(idx))
        return x + 0.015 * np.random.randn(N)
    else:           # Avion
        freqs = np.fft.rfftfreq(N, 1/FS)
        asd = np.zeros_like(freqs)
        m = freqs >= 1
        asd[m] = 5e-4 * (1 + (freqs[m]/10)**0.5)
        sp = np.sqrt(asd * N / (2*FS)) * np.exp(1j * 2*np.pi*np.random.rand(len(freqs)))
        x = np.fft.irfft(sp, N)
        return x * (0.15 / (x.std() + 1e-12))

def extract_6(x):
    grms  = np.sqrt(np.mean(x**2)) + 1e-12
    crete = np.max(np.abs(x)) / grms
    fw, pxx = welch(x, fs=FS, nperseg=128, noverlap=64)
    def band(f1, f2):
        m = (fw >= f1) & (fw <= f2)
        return float(np.trapezoid(pxx[m], fw[m])) if m.any() else 0.0
    etot   = band(1, FS/2) + 1e-20
    eb     = band(1, 5)   / etot
    em     = band(5, 15)  / etot
    eh     = band(15, FS/2)/etot
    mc     = (fw >= 1) & (fw <= FS/2)
    fc     = float((fw[mc] * pxx[mc]).sum() / (pxx[mc].sum() + 1e-20))
    return [grms, crete, eb, em, eh, fc]

X6, y6 = [], []
for label in range(4):
    for _ in range(100):
        X6.append(extract_6(gen_signal(label, N_WIN)))
        y6.append(label)
X6 = np.array(X6)
y6 = np.array(y6)

# k-NN avec normalisation
from sklearn.pipeline import make_pipeline
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3))
scores = cross_val_score(pipe, X6, y6, cv=5, scoring='accuracy')

print('k-NN (k=3), 6 descripteurs, 4 classes, validation croisée 5 plis :')
print(f'  Précision moyenne : {scores.mean()*100:.1f} %  ±  {scores.std()*100:.1f} %')
print()
print('Valeurs moyennes des 6 descripteurs par classe (avant normalisation) :')
noms = ['g_rms', 'Crete', 'E_bas', 'E_mil', 'E_hau', 'f_c (Hz)']
print(f'  {"Classe":<8}  ' + '  '.join(f'{n:>10}' for n in noms))
for label, mode in enumerate(MODES_4):
    m = y6 == label
    vals = X6[m].mean(axis=0)
    print(f'  {mode:<8}  ' + '  '.join(f'{v:>10.4f}' for v in vals))

## Conclusion

| Situation | Résultat | Raison |
|---|---|---|
| Sans normalisation | **Train** (incorrect) | L'axe $f_c$ (Hz) représente >95 % de la distance au carré |
| Avec normalisation | **Camion** (correct)  | Les deux axes ont le même poids ($\sigma = 1$) |

**Règle générale :** avant tout calcul de distance euclidienne dans un k-NN,
**toujours normaliser** (centrage-réduction) avec les paramètres du jeu d'entraînement.
Appliquer ensuite les **mêmes** paramètres au vecteur inconnu.

En sklearn :
```python
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3))
pipe.fit(X_train, y_train)
pred = pipe.predict(x_inconnu.reshape(1, -1))
```